# 提示词模板

In [3]:
import os

import dotenv
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ChatMessage
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()

m = ChatOpenAI(
    model='qwen-plus',
    api_key=os.getenv('ALI_API_KEY'),
    base_url=os.getenv('ALI_API_URL'),
)

prompt_template = PromptTemplate(template="你是一个{role},你的名字叫{name}",input_variables=["role", "name"])

fmt = prompt_template.format(role="有帮助的助手", name="AIBOT")
print(fmt)

你是一个有帮助的助手,你的名字叫AIBOT


In [4]:
from langchain_core.prompts import PromptTemplate
prompt_template = PromptTemplate.from_template(template="你是一个{role},你的名字叫{name}")

fmt = prompt_template.format(role="有帮助的助手", name="AIBOT")
print(fmt)

你是一个有帮助的助手,你的名字叫AIBOT


## 2. 两个特殊结构的使用（部分提示词的使用、组合提示的使用）

### 2.1 部分提示词的使用重点

partial_variables 参数可以直接传入字典

In [7]:
from langchain_core.prompts import PromptTemplate
prompt_template = PromptTemplate.from_template(template="你是一个{role},你的名字叫{name}, aspect1:{aspect1}",
                                               partial_variables={
                                                   "aspect1":"有帮助的助手"
                                               })

fmt = prompt_template.format(role="有帮助的助手", name="AIBOT")
print(fmt)

你是一个有帮助的助手,你的名字叫AIBOT, aspect1:有帮助的助手


In [18]:
from langchain_core.prompts import PromptTemplate
prompt_template = PromptTemplate.from_template(template="你是一个{role},你的名字叫{name}, aspect1:{aspect1}",
                                              ).partial(role="有帮助的助手", name="AIBOT",)

fmt = prompt_template.format(aspect1="xxx")
print(fmt)

你是一个有帮助的助手,你的名字叫AIBOT, aspect1:xxx


### 组合提示词

In [20]:
from langchain_core.prompts import PromptTemplate
prompt_template = PromptTemplate.from_template(template="你是一个{role},你的名字叫{name}, aspect1:{aspect1}",
                                              ).partial(role="有帮助的助手", name="AIBOT",)
template = (prompt_template + "\n 这是追加的内容：{additional_info}").partial(additional_info="这是额外的信息")
fmt = template.format(aspect1="xxx")
print(fmt)

你是一个有帮助的助手,你的名字叫AIBOT, aspect1:xxx
 这是追加的内容：这是额外的信息


# 给变量复制的两种方式
format invoke

format 使用参数的方式来实现

invoke 使用 字典 参数名字是 input 来自赋值字符串模板


In [27]:
from langchain_core.prompts import PromptTemplate
prompt_template = PromptTemplate.from_template(template="你是一个{role},你的名字叫{name}, aspect1:{aspect1}",).partial(role="有帮助的助手", name="AIBOT",)
# fmt = prompt_template.format(aspect1="xxx")
fmt = prompt_template.invoke(input={"aspect1":"xxx"})
print(fmt)

text='你是一个有帮助的助手,你的名字叫AIBOT, aspect1:xxx'


In [30]:
import os

import dotenv
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ChatMessage
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()

m = ChatOpenAI(
    model='qwen-plus',
    api_key=os.getenv('ALI_API_KEY'),
    base_url=os.getenv('ALI_API_URL'),
)
prompt_template = PromptTemplate.from_template(template="你是一个{role},你的名字叫{name}, 你的每一次回答都只能回答{num}个文字",).partial(role="有帮助的助手", name="AIBOT",)
# fmt = prompt_template.format(aspect1="xxx")
fmt = prompt_template.invoke(input={"num":"50"})
print(fmt)
resp = m.invoke(fmt)
print(resp.content)

text='你是一个有帮助的助手,你的名字叫AIBOT, 你的每一次回答都只能回答50个文字'
你好，我是AIBOT，很高兴为你服务。请告诉我你需要什么帮助，我会尽力在50字内为你解答。


# chat_prompt 消息模板使用

1. 实例化 (构造， from_messages())
2. 调用提示词模板的几种方法： invoke() \ format() \ format_message() \ format_prompt()
3. 更丰富的实例化参数类型
4. 结合 LLM
5. 插入消息列表:MessagePlaceholder


 ### 实例化 - 构造

In [34]:
import os

import dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()

client = ChatOpenAI(
    model='qwen-plus',
    api_key=os.getenv('ALI_API_KEY'),
    base_url=os.getenv('ALI_API_URL'),
)
chat_prompt_template = ChatPromptTemplate(
    messages=[
        ("system","你是一个ai助手，你的名字叫做{name}"),
        ("human","我的问题是：{question}")
    ],
    input_variables=[],)
cpt = chat_prompt_template.invoke(input={
    "name":"ABBOT",
    "question":"你是谁？"
})
print(cpt)

messages=[SystemMessage(content='你是一个ai助手，你的名字叫做ABBOT', additional_kwargs={}, response_metadata={}), HumanMessage(content='我的问题是：你是谁？', additional_kwargs={}, response_metadata={})]


### 使用 from_messages()

In [45]:
import os

import dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()

client = ChatOpenAI(
    model='qwen-plus',
    api_key=os.getenv('ALI_API_KEY'),
    base_url=os.getenv('ALI_API_URL'),
)
chat_prompt_template = ChatPromptTemplate.from_messages([
        ("system","你是一个ai助手，你的名字叫做{name}"),
        ("human","我的问题是：{question}")
    ])
print("==================================== invoke ====================================")
cpt = chat_prompt_template.invoke(input={
    "name":"ABBOT",
    "question":"你是谁？"
})
print(cpt)
# format
print("==================================== format ====================================")
cpt = chat_prompt_template.format(name="xxxx",question="你是谁？")
print(cpt)
# format_messages,返回类型不一样，发挥 list
print("==================================== format_messages ====================================")
cpt = chat_prompt_template.format_messages(name="xxxx",question="你是谁？")
print(cpt)
resp = client.invoke(cpt)

print(resp.content)
# format_prompt,返回类型不一样，返回  ChatPromptValue
print("==================================== format_prompt ====================================")
cpt = chat_prompt_template.format_prompt(name="xxxx",question="你是谁？")
print(cpt)

resp = client.invoke(cpt)

print(resp.content)

==================================== invoke ====================================
messages=[SystemMessage(content='你是一个ai助手，你的名字叫做ABBOT', additional_kwargs={}, response_metadata={}), HumanMessage(content='我的问题是：你是谁？', additional_kwargs={}, response_metadata={})]
==================================== format ====================================
System: 你是一个ai助手，你的名字叫做xxxx
Human: 我的问题是：你是谁？
==================================== format_messages ====================================
[SystemMessage(content='你是一个ai助手，你的名字叫做xxxx', additional_kwargs={}, response_metadata={}), HumanMessage(content='我的问题是：你是谁？', additional_kwargs={}, response_metadata={})]
我是Qwen，是阿里云研发的超大规模语言模型。我能够回答问题、创作文字，比如写故事、写公文、写邮件、写剧本等等，还能进行逻辑推理、编程等任务。不过您提到的名字“xxxx”似乎是一个占位符，我的正确名称是Qwen。如果您有任何问题或需要帮助，欢迎随时告诉我！
==================================== format_prompt ====================================
messages=[SystemMessage(content='你是一个ai助手，你的名字叫做xxxx', additional_kwargs={}, response_metadata={}), HumanMessage(content='我的问题是：你是谁？'

# MessagesPlaceholder

In [46]:
import os

import dotenv
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()

client = ChatOpenAI(
    model='qwen-plus',
    api_key=os.getenv('ALI_API_KEY'),
    base_url=os.getenv('ALI_API_URL'),
)
chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system","你是一个ai助手，你的名字叫做{name}"),
    MessagesPlaceholder(variable_name="msgs")
])

print(resp.content)
# format_prompt,返回类型不一样，返回  ChatPromptValue
print("==================================== format_prompt ====================================")
cpt = chat_prompt_template.format_messages(
    name=" xxxx ",
    # 由于使用了消息placeholder 可以传入消息列表
   msgs=[HumanMessage(content="我的问题是：1+2=？")])
print(cpt)

resp = client.invoke(cpt)

print(resp.content)

我是Qwen，是阿里云研发的超大规模语言模型。我被设计用来协助用户撰写故事、诗歌、公文，回答问题和提供其他类型的文本创作帮助。不过你提到的名字“xxxx”似乎是一个占位符，我的正确名字是Qwen。有什么我可以帮你的吗？
==================================== format_prompt ====================================
[SystemMessage(content='你是一个ai助手，你的名字叫做xxxx', additional_kwargs={}, response_metadata={}), HumanMessage(content='我的问题是：1+2=？', additional_kwargs={}, response_metadata={})]
1 + 2 = 3。
